# Per-Night Polarized Source Filtering Notebook

**by Tyler Cox**, last updated March 26, 2025

This notebook fits polarized point source models (position + rotation measure) from per-night HERA single-baseline data, then builds Faraday-rotating DPSS models suitable for subtracting those sources from the visibilities. For each source in the catalog, it:

1. Loads pseudo-Stokes-Q visibilities over a narrow LST window around each source.
2. Iteratively fits the source's sky position and rotation measure.
3. Forms a baseline-averaged, phase-rotated dataset for all sources.
4. Fits a separable DPSS model (in time and frequency) to the Faraday-rotated signal.

Here's a set of links to skip to particular figures and tables:
# [• Figure 1: Faraday Depth Spectra](#Figure-1:-Faraday-Depth-Spectra)
# [• Figure 2: Source Images and PSF Contours](#Figure-2:-Source-Images-and-PSF-Contours)
# [• Figure 3: Polarized Source Model vs. Data (Low Band)](#Figure-3:-Polarized-Source-Model-vs.-Data-(Low-Band))
# [• Figure 4: Polarized Source Model vs. Data (High Band)](#Figure-4:-Polarized-Source-Model-vs.-Data-(High-Band))


In [ ]:
import time
tstart = time.time()
!hostname
!date

In [ ]:
import os
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'

import glob
import warnings
from multiprocessing import Pool

import h5py
import numpy as np
import matplotlib.pyplot as plt
from astropy import constants
from astropy.coordinates import SkyCoord
from astropy.stats import mad_std
from scipy import optimize
from tqdm.notebook import tqdm

from pyuvdata import Telescope, UVBeam, UVData
from hera_cal import frf, io, redcal, utils, polfilt, datacontainer, flag_utils
from hera_cal.smooth_cal import _linear_fit
from hera_filters import dspec
import uvtools
import snapshot_imager

import yaml
from pyuvdata import UVFlag

from IPython.display import display
%matplotlib inline

## Load Files for Polarized Source Modeling

In [ ]:
RED_AVG_FILE = os.environ.get("RED_AVG_FILE", None)
# RED_AVG_FILE = '/lustre/aoc/projects/hera/h6c-analysis/IDR3_2/2459930/zen.2459930.21324.sum.smooth_calibrated.red_avg.uvh5'

CORNER_TURN_MAP_YAML = os.environ.get("CORNER_TURN_MAP_YAML", 
                                        os.path.join(os.path.dirname(RED_AVG_FILE), "single_baseline_files/corner_turn_map.yaml"))

PRIOR_FLAG_SUFFIX = os.environ.get("PRIOR_FLAG_SUFFIX", ".flag_waterfall_round_3.h5")

# Source catalogue.  Each entry specifies an initial RA (deg), Dec (deg), RM, scintillation scale, etc.
SOURCE_YAML = os.environ.get(
    "SOURCE_YAML", "/lustre/aoc/projects/hera/h6c-analysis/IDR3_2/metadata/sources.yaml"
)

# Minimum number of times necessary to perform modeling
MIN_TIMES = int(os.environ.get("MIN_TIMES", 50))

# Frequency channels to include when fitting source parameters (channel indices).
FREQ_CHAN_START = int(os.environ.get("FREQ_CHAN_START", 500))
FREQ_CHAN_END   = int(os.environ.get("FREQ_CHAN_END",  1500))

# Maximum allowable difference from catalog values before falling back to catalog values
MAX_RM_DIFF = float(os.environ.get("MAX_RM_DIFF", 2.0)) # rad / m^2
MAX_RA_DIFF = float(os.environ.get("MAX_RA_DIFF", 1.0)) # degrees
MAX_DEC_DIFF = float(os.environ.get("MAX_DEC_DIFF", 1.0)) # degrees
MIN_SNR = float(os.environ.get("MIN_SNR", 5.0))

# FM band edges used to split the high-band into sub-bands for filtering.
FM_LOW_FREQ  = float(os.environ.get("FM_LOW_FREQ",  87.5))   # MHz
FM_HIGH_FREQ = float(os.environ.get("FM_HIGH_FREQ", 108.0))  # MHz

# Delay filter half-width (seconds) and DPSS eigenvalue cutoff.
FG_DELAY_HALF_WIDTH = float(os.environ.get("FG_DELAY_HALF_WIDTH",   500e-9))  # seconds
MODEL_DELAY_HALF_WIDTH = float(os.environ.get("MODEL_DELAY_HALF_WIDTH",   50e-9)) # seconds
MODEL_FRATE_HALF_WIDTH = float(os.environ.get("MODEL_FRATE_HALF_WIDTH",   0.25)) # mHz
EIGENVAL_CUTOFF = float(os.environ.get("EIGENVAL_CUTOFF", 1e-12))

# Maximum number of source-parameter fitting iterations.
FIT_MAXITER = int(os.environ.get("FIT_MAXITER", 10))

# Number of parallel workers for the multiprocessing file loader.
MAX_WORKERS = int(os.environ.get("MAX_WORKERS", 8))

SAVE_MODELS = os.environ.get("SAVE_MODELS", "TRUE").upper() == "TRUE"

# Print all settings
for setting in ['RED_AVG_FILE', 'CORNER_TURN_MAP_YAML', 'PRIOR_FLAG_SUFFIX', 'SOURCE_YAML']:
    print(f'{setting} = "{eval(setting)}"')

for setting in ['MIN_TIMES', 'FREQ_CHAN_START', 'FREQ_CHAN_END',
                'MAX_RM_DIFF', 'MAX_RA_DIFF', 'MAX_DEC_DIFF', 'MIN_SNR',
                'FM_LOW_FREQ', 'FM_HIGH_FREQ',
                'FG_DELAY_HALF_WIDTH', 'MODEL_DELAY_HALF_WIDTH', 'MODEL_FRATE_HALF_WIDTH',
                'EIGENVAL_CUTOFF', 'FIT_MAXITER', 'MAX_WORKERS', 'SAVE_MODELS']:
    print(f'{setting} = {eval(setting)}')


In [ ]:
add_to_history = 'Produced by per_night_source_filtering notebook with the following environment:\n' + '=' * 65 + '\n' + os.popen('mamba env export').read() + '=' * 65

## Preliminaries

In [ ]:
with open(CORNER_TURN_MAP_YAML, 'r') as file:
    corner_turn_map = yaml.unsafe_load(file)

jdstr = [s for s in os.path.basename(RED_AVG_FILE).split('.') if s.isnumeric()][0]
flag_dir = os.path.dirname(CORNER_TURN_MAP_YAML)
flag_pattern = os.path.join(flag_dir, f'zen.{jdstr}*{PRIOR_FLAG_SUFFIX}')
prior_flag_files = sorted(glob.glob(flag_pattern))

if len(prior_flag_files) == 0:
    raise ValueError(f'APPLY_PRIOR_FLAGS is True but no files matched {flag_pattern}')
else:
    print(f'Found {len(prior_flag_files)} prior flag files:')
    for f in prior_flag_files:
        print(f'  {os.path.basename(f)}')
    prior_flags = np.any([np.all(UVFlag(flag_file).flag_array, axis=-1) for flag_file in prior_flag_files], axis=0)
    print(f'Combined prior flags: {np.mean(prior_flags):.3%} flagged.')

In [ ]:
# Discover single-baseline files, sorted by redundant-group size (largest first)
# so that the most constrained baselines are processed first.
load_files = sorted(glob.glob(os.path.join(os.path.dirname(CORNER_TURN_MAP_YAML), "*.inpainted.uvh5")))
print(f"Found {len(load_files)} reinpainted single-baseline files under {os.path.dirname(CORNER_TURN_MAP_YAML)}")

# Drop autocorrelation files -- baselines are encoded as "ANT1_ANT2" in the filename
baselines = [
    tuple(map(int, os.path.basename(f).split(".")[3].split("_")))
    for f in load_files
]
auto_file = [f for (ai, aj), f in zip(baselines, load_files) if ai == aj]
load_files = [f for (ai, aj), f in zip(baselines, load_files) if ai != aj]
print(f'{len(load_files)} cross-correlation files after removing autocorrelations.')

## Identify Which Sources Are Within Range

In [ ]:
# Load in source file
with open(SOURCE_YAML) as f:
    data = yaml.safe_load(f)

source_list = data["sources"]

# Find sources that have enough times to fit
hd_meta = io.HERAData(load_files[0])
sources = {}
for si, source in enumerate(source_list):
    source_name = source["name"]
    source_coord = SkyCoord(ra=source["right_ascension"], dec=source["declination"])
    
    lst_center = source_coord.ra.deg / 15 * np.pi / 12
    lst_mask = (
        (hd_meta.lsts > lst_center - source["lst_half_width"] * np.pi / 12) &
        (hd_meta.lsts < lst_center + source["lst_half_width"] * np.pi / 12)
    )
    if lst_mask.sum() > MIN_TIMES:
        sources[source_name] = {
            "right_ascension":  source_coord.ra.deg,
            "declination":      source_coord.dec.deg,
            "rotation_measure": source["rotation_measure"],
            "fit_scintillation": source["fit_scintillation"],
            "scintillation_timescale": source["scintillation_timescale"],
            "lst_half_width": source["lst_half_width"] * np.pi / 12
        }
        print(f'{source_name}: {lst_mask.sum()} integrations in window -- INCLUDED')
    else:
        print(f'{source_name}: {lst_mask.sum()} integrations in window -- SKIPPED (< {MIN_TIMES})')

## Load Pseudo-Stokes-Q Data Around Each Source

In [ ]:
FREQ_SLICE: np.ndarray = np.arange(FREQ_CHAN_START, FREQ_CHAN_END) # JUST USE THE HIGH BAND FOR LOCALIZATION

In [ ]:
def closest_values(v, p, k=100):
    """Return ``(start, end)`` indices of the *k*-element window in *v* centred
    nearest to value *p*.

    Parameters
    ----------
    v : array_like
        Sorted 1-D array to search.
    p : float
        Target value.
    k : int
        Window length (default 100).

    Returns
    -------
    start, end : int
        Slice indices such that ``v[start:end]`` is the closest *k*-element
        segment to *p*.
    """
    idx = np.searchsorted(v, p)
    half = k // 2
    start = max(0, idx - half)
    end = start + k
    if end > len(v):
        end = len(v)
        start = end - k
    return start, end

In [ ]:
def _process_file_hera(filepath):
    """Read a single HERA file and extract Stokes-Q visibilities for every source.

    For each source defined in the global ``sources`` dict the function:

    1. Reads the ``ee`` and ``nn`` polarisations over a narrow LST window
       centred on the source's right ascension.
    2. Forms the pseudo-Stokes-Q visibility ``pQ = ee - nn`` for every
       baseline.
    3. Combines flags with a logical OR and propagates inverse-variance
       sample weights (non-finite values are zeroed out).

    Parameters
    ----------
    filepath : str
        Path to an inpainted HERA UVH5 data file.

    Returns
    -------
    out_data, out_flags, out_nsamples : tuple of lists
        Three parallel lists (one entry per source) of baseline-keyed dicts
        containing pQ visibilities, combined flags, and harmonic-mean sample
        weights respectively.  Returns ``None`` if all sources fail to load.
    """
    hd = io.HERAData(filepath)

    out_data     = [{} for _ in sources]
    out_flags    = [{} for _ in sources]
    out_nsamples = [{} for _ in sources]

    for si, source in enumerate(sources):
        lst_center = sources[source]["right_ascension"] / 15 * np.pi / 12
        imin, imax = closest_values(hd.lsts, lst_center, k=MIN_TIMES)
        lst_range  = (hd.lsts[imin], hd.lsts[imax - 1])

        try:
            d, f, n = hd.read(
                polarizations=["ee", "nn"],
                lst_range=lst_range,
                freq_chans=FREQ_SLICE,
            )
        except Exception:
            continue

        for ap in d.antpairs():
            ee  = d[ap + ("ee",)];  nn  = d[ap + ("nn",)]
            fee = f[ap + ("ee",)] | prior_flags[imin:imax][:, FREQ_SLICE];  fnn = f[ap + ("nn",)] | prior_flags[imin:imax][:, FREQ_SLICE]
            nee = n[ap + ("ee",)];  nnn = n[ap + ("nn",)]

            # Pseudo-Stokes Q
            out_data[si][ap + ("pQ",)] = ee - nn

            # Union of per-polarisation flags
            out_flags[si][ap + ("pQ",)] = fee | fnn

            # Harmonic-mean sample count; replace non-finite entries with 0
            with np.errstate(divide="ignore", invalid="ignore"):
                nv = (nee**-1.0 + nnn**-1.0) ** -1.0
            out_nsamples[si][ap + ("pQ",)] = np.where(np.isfinite(nv), nv, 0.0)

    return out_data, out_flags, out_nsamples


def load_parallel(files, max_workers=16):
    """Load and merge HERA data across multiple files in parallel.

    Files are processed concurrently using :class:`multiprocessing.Pool`.
    After all workers finish, metadata (antenna positions, frequencies, LST
    timestamps) is attached by re-reading the first file serially.

    Parameters
    ----------
    files : list of str
        Ordered list of file paths to load.  The first entry is used as the
        metadata reference.
    max_workers : int
        Number of worker processes in the pool (default: 16).

    Returns
    -------
    data_list, flags_list, nsamples_list : dict of DataContainer
        Pseudo-Stokes-Q visibilities, combined flags, and inverse-variance
        sample weights, each keyed by source name.
    """
    data     = [{} for _ in sources]
    flags    = [{} for _ in sources]
    nsamples = [{} for _ in sources]

    with Pool(processes=max_workers) as pool:
        results = pool.imap_unordered(_process_file_hera, files, chunksize=1)
        for result in tqdm(results, total=len(files)):
            if result is None:
                continue
            try:
                out_data, out_flags, out_nsamples = result
                for si in range(len(sources)):
                    data[si].update(out_data[si])
                    flags[si].update(out_flags[si])
                    nsamples[si].update(out_nsamples[si])
            except Exception as exc:
                print(f"Merge error: {exc}")
                raise

    # Attach metadata from the first file and wrap in DataContainers
    hd = io.HERAData(files[0])
    data_list     = {}
    flags_list    = {}
    nsamples_list = {}

    for si, source_name in enumerate(sources):
        lst_center = sources[source_name]["right_ascension"] / 15 * np.pi / 12
        imin, imax = closest_values(hd.lsts, lst_center, k=MIN_TIMES)
        lst_range  = (hd.lsts[imin], hd.lsts[imax - 1])

        try:
            d, _, _ = hd.read(
                polarizations=["ee", "nn"],
                lst_range=lst_range,
                freq_chans=FREQ_SLICE,
            )
        except Exception:
            continue

        dc = datacontainer.DataContainer(data[si])
        dc.antpos = d.antpos
        dc.freqs  = d.freqs
        dc.times  = d.times

        data_list[source_name]     = dc
        flags_list[source_name]    = datacontainer.DataContainer(flags[si])
        nsamples_list[source_name] = datacontainer.DataContainer(nsamples[si])

    return data_list, flags_list, nsamples_list

In [ ]:
data_list, flags_list, nsamples_list = load_parallel(
    load_files, max_workers=MAX_WORKERS
)

## Fit Source Positions and Rotation Measures

In [ ]:
"""
Fitting utilities for polarized source parameters (RA, Dec, Rotation Measure).

Provides functions to unpack HERA visibility data and iteratively fit the
sky position and Faraday rotation measure of polarized point sources.
"""
def plot_source_rotation_measure(data, flags, nsamples, ra, dec, rotation_measure,
                                  location, drm=5, dtest=500):
    """
    Evaluate the Faraday depth spectrum around a source.

    Phases all pQ visibilities to ``(ra, dec)``, averages into a single
    spectrum, and sweeps a grid of rotation measures to build the Faraday
    dispersion function.

    Parameters
    ----------
    data, flags, nsamples : datacontainer.DataContainer
        pQ visibilities.
    ra, dec : float
        Best-fit source position in degrees.
    rotation_measure : float
        Best-fit RM (used as the centre of the search window).
    location : astropy.coordinates.EarthLocation
        Observatory location.
    drm : float, optional
        Half-width of the RM search window in rad/m². Default is 5.
    dtest : int, optional
        Number of trial RM values. Default is 500.

    Returns
    -------
    test_rm : np.ndarray, shape (dtest,)
        Trial RM values in rad/m².
    faraday_response : np.ndarray, shape (dtest,)
        Amplitude of the de-rotated spectrum at each trial RM.
    """
    vis, weights, uvw, times, freqs = polfilt.unpack_data_containers(
        data, flags, nsamples, pol='pQ'
    )
    weights = np.where(np.isfinite(vis), weights, 0.0)
    vis = np.where(np.isfinite(vis), vis, 0.0)

    # Phase-rotate to source position and compute a weighted-average spectrum.
    l, m, n = polfilt.radec_to_azalt(ra, dec, times, location)
    lmn = np.array([l, m, n])  # shape (3, n_times)
    phasor = np.exp(-2j * np.pi * np.einsum("bcf,ct->btf", uvw, lmn))

    vis_phased = vis * phasor  # (n_bls, n_times, n_freqs)
    weight_sum = np.sum(weights, axis=(0, 1))  # (n_freqs,)
    safe_weight_sum = np.where(weight_sum > 0, weight_sum, 1.0)
    spectrum = (
        np.sum(vis_phased * weights, axis=(0, 1)) / safe_weight_sum
    )  # (n_freqs,)

    # Grid search over RM
    lambda_sq = (constants.c.value / freqs) ** 2
    test_rm = np.linspace(rotation_measure - drm, rotation_measure + drm, dtest)
    faraday_response = np.array(
        [
            np.abs(np.nanmean(spectrum * np.exp(2j * lambda_sq * rm)))
            for rm in test_rm
        ]
    )

    return test_rm, faraday_response

def image_polarized_data(data, flags, nsamples, ra, dec, rotation_measure, location, npix=300, fov=20, verbose=False):
    """Form a dirty image of the polarized sky around a source.

    Phase-rotates all baselines to the source position, applies the Faraday
    phasor ``exp(2i lambda^2 RM)`` to de-rotate in frequency, and accumulates a
    weighted 2-D DFT image on a flat-sky ``(l, m)`` grid.

    Parameters
    ----------
    data, flags, nsamples : DataContainer
        Pseudo-Stokes-Q visibilities, flags, and weights for the source window.
    ra, dec : float
        Phase-centre right ascension and declination in degrees.
    rotation_measure : float
        Source rotation measure in rad/m^2.
    dl, dm : array_like
        1-D flat-sky direction cosine offsets (radians) for the image grid.
    location : EarthLocation
        Telescope location used by ``polfilt.radec_to_azalt``.

    Returns
    -------
    image : ndarray, complex, shape (ngrid, ngrid)
        Normalised dirty image.
    psf : ndarray, real, shape (ngrid, ngrid)
        Normalised point-spread function (real part of the weights-only image).
    """
    imaging_data = snapshot_imager.unpack_data_containers(
        data, flags, nsamples, pol="pQ"
    )
    weights = np.where(np.isfinite(imaging_data.vis), imaging_data.weights, 0.0)
    vis     = np.where(np.isfinite(imaging_data.vis), imaging_data.vis,     0.0)
    
    # Get phasors
    lambda_sq = (constants.c.value / imaging_data.freqs) ** 2
    rm_phasor = np.exp(2j * lambda_sq * rotation_measure)  # (n_freqs,)
    l0, m0, n0 = polfilt.radec_to_azalt(ra, dec, imaging_data.times, location)
    lmn = np.array([l0, m0, n0])       # (3,)
    phase0 = np.einsum("bcf,ct->btf", imaging_data.uvw, lmn)                      # (n_freqs,)
    
    vis_phased = (
        vis
        * np.exp(-2j * np.pi * phase0) * weights
    )
    vis_avg = np.mean(vis_phased, axis=1, keepdims=True)
    w_avg = np.mean(weights, axis=1, keepdims=True)
    vis_avg /= w_avg
    vis_avg = np.where(np.isfinite(vis_avg), vis_avg, 0.0)

    # Put in format for snapshot imager
    imaging_data.vis = vis_avg
    imaging_data.weights = w_avg
    imaging_data.times = np.mean(imaging_data.times)
    images = snapshot_imager.snapshot_imager_type1(
        imaging_data, 
        fov=fov, 
        npix=npix, 
        rm_phasor=rm_phasor,
        verbose=verbose
    )
    return images

In [ ]:
# Build a monotonically increasing LST array, unwrapping across midnight if needed
lst_array = hd_meta.lsts.copy()
if lst_array[0] > lst_array[-1]:
    lst_midpoint = (lst_array[0] + lst_array[-1]) / 2
    lst_array[lst_array > lst_midpoint] -= 2 * np.pi

In [ ]:
best_fit_source_params = {}
source_images = []

for source_name in data_list:
    lst_center = sources[source_name]['right_ascension'] / 15 * np.pi / 12
    imin, imax = closest_values(hd_meta.lsts, lst_center, k=MIN_TIMES)

    fit_ra, fit_dec, fit_rm = polfilt.iteratively_fit_polarized_source_params(
        data_list[source_name],
        flags_list[source_name],
        nsamples_list[source_name],
        sources[source_name]['right_ascension'],
        sources[source_name]['declination'],
        sources[source_name]['rotation_measure'],
        hd_meta.telescope.location,
        maxiter=FIT_MAXITER,
        method='scipy',
        verbose=False,
    )

    # Use fit parameters to check image to evaluate fit
    images = image_polarized_data(
        data_list[source_name],
        flags_list[source_name],
        nsamples_list[source_name],
        fit_ra, 
        fit_dec,
        fit_rm,
        hd_meta.telescope.location,
    )
    avg_image = np.squeeze(images.images)
    sigma = np.sqrt((mad_std(avg_image.real) ** 2 + mad_std(avg_image.imag) ** 2) / 2)
    central_pixel = np.abs(avg_image[avg_image.shape[0] // 2, avg_image.shape[1] // 2])
    
    # Store the image
    source_images.append(avg_image)

    # Compute the difference between the fit parameters and catalog ra/dec/rm
    ra_diff = np.abs(fit_ra - sources[source_name]["right_ascension"])
    dec_diff = np.abs(fit_dec - sources[source_name]["declination"])
    rm_diff = np.abs(fit_rm - sources[source_name]["rotation_measure"])
    
    # Check that the phase RM 
    if (
        np.abs(central_pixel) / sigma > MIN_SNR
        and (ra_diff < MAX_RA_DIFF)
        and (dec_diff < MAX_DEC_DIFF)
        and (rm_diff < MAX_RM_DIFF)
    ):
        print(
            f'{source_name}: RA={fit_ra:.4f} deg (delta={fit_ra - sources[source_name]["right_ascension"]:+.4f} deg), '
            f'Dec={fit_dec:.4f} deg (delta={fit_dec - sources[source_name]["declination"]:+.4f} deg), '
            f'RM={fit_rm:.2f} rad/m^2'
        )
        best_fit_source_params[source_name] = {
            "right_ascension":  fit_ra,
            "declination":      fit_dec,
            "rotation_measure": fit_rm,
            "fit_scintillation": sources[source_name]["fit_scintillation"],
            "scintillation_timescale": sources[source_name]["scintillation_timescale"],
            "lst_half_width": sources[source_name]["lst_half_width"]
        }
    else:
        print (
            source_name,
            "\n\tFalling back to catalog values",
            "\n\tSNR:", np.abs(central_pixel) / sigma, 
            "\n\tdelta ra:", (ra_diff), 
            "\n\tdelta dec:", (dec_diff), 
            "\n\tdelta rm:", (rm_diff)
        )
        best_fit_source_params[source_name] = {
            "right_ascension":  sources[source_name]["right_ascension"],
            "declination":      sources[source_name]["declination"],
            "rotation_measure": sources[source_name]["rotation_measure"],
            "fit_scintillation": sources[source_name]["fit_scintillation"],
            "scintillation_timescale": sources[source_name]["scintillation_timescale"],
            "lst_half_width": sources[source_name]["lst_half_width"]
        }

In [ ]:
source_images = []
for source_name in best_fit_source_params:
    lst_center = sources[source_name]['right_ascension'] / 15 * np.pi / 12
    imin, imax = closest_values(hd_meta.lsts, lst_center, k=MIN_TIMES)
    
    images = image_polarized_data(
        data_list[source_name],
        flags_list[source_name],
        nsamples_list[source_name],
        best_fit_source_params[source_name]["right_ascension"], 
        best_fit_source_params[source_name]["declination"],
        best_fit_source_params[source_name]["rotation_measure"],
        hd_meta.telescope.location,
        fov=2,
        npix=100
    )
    source_images.append(np.squeeze(images.images))

In [ ]:
faraday_depth = []
test_rm_grid  = []

for source_name in best_fit_source_params:
    lst_center = sources[source_name]['right_ascension'] / 15 * np.pi / 12
    imin, imax = closest_values(
        hd_meta.lsts, 
        lst_center, 
        k=MIN_TIMES
    )

    _rm, _fd = plot_source_rotation_measure(
        data_list[source_name],
        flags_list[source_name],
        nsamples_list[source_name],
        best_fit_source_params[source_name]["right_ascension"],
        best_fit_source_params[source_name]["declination"],
        best_fit_source_params[source_name]["rotation_measure"],
        hd_meta.telescope.location,
    )
    test_rm_grid.append(_rm)
    faraday_depth.append(_fd)

# *Figure 1: Faraday Depth Spectra*

In [ ]:
source_names_fit = list(best_fit_source_params.keys())
n_sources        = len(source_names_fit)

for si, source_name in enumerate(source_names_fit):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(test_rm_grid[si], faraday_depth[si])
    ax.axvline(sources[source_name]["rotation_measure"], color='k',
               linestyle='--', label=f'Catalog RM: {np.round(sources[source_name]["rotation_measure"], 2)}')
    ax.axvline(best_fit_source_params[source_name]["rotation_measure"], color='r',
               linestyle='--', label=f'Fit RM: {np.round(best_fit_source_params[source_name]["rotation_measure"], 2)}')
    ax.set_xlabel('Rotation Measure (rad/m^2)')
    ax.set_ylabel('Faraday Response')
    ax.set_title(source_name)
    ax.legend()
    plt.tight_layout()
    display(fig)
    plt.close(fig)

# *Figure 2: Source Images and PSF Contours*

In [ ]:
for si, source_name in enumerate(source_names_fit):
    ra  = best_fit_source_params[source_name]["right_ascension"]
    dec = best_fit_source_params[source_name]["declination"]
    extent = [ra - 1, ra + 1, dec - 1, dec + 1]
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(
        np.abs(source_images[si])[::-1], cmap='turbo',
        interpolation='None', aspect='auto', extent=extent,
    )
    ax.set_title(source_name)
    ax.set_xlabel('Right Ascension (deg)')
    ax.set_ylabel('Declination (deg)')

    ax.scatter(
        sources[source_name]['right_ascension'], sources[source_name]['declination'],
        color='white', marker='o', 
        label=f'Catalog position (RA, DEC):' \
              f'({np.round(sources[source_name]["right_ascension"], 2)}, {np.round(sources[source_name]["declination"], 2)})',
    )
    ax.scatter(
        ra, dec,
        color='white', marker='x', s=80, 
        label='Fit position (RA, DEC):' \
             f'({np.round(ra, 2)}, {np.round(dec, 2)})',
    )
    ax.legend(fontsize=8)
    plt.tight_layout()
    display(fig)
    plt.close(fig)

## Build Faraday-Rotating DPSS Source Models

With best-fit positions and rotation measures in hand, we now load a wider LST window (+/-1 hour) of data, average over all baselines (phase-rotating each to the source), and fit a separable DPSS model in time x frequency that captures the Faraday-modulated source signal.

In [ ]:
def _process_file_hera_with_phasor(filepath):
    """Phase-rotate and accumulate weighted visibilities for each source.

    For every source in ``best_fit_source_params`` the function reads the
    full +/-1-hour LST window, computes the geometric phase for the baseline
    toward the source at each time/frequency sample, and returns per-source
    weighted numerators and denominators ready for averaging across baselines.

    Parameters
    ----------
    filepath : str
        Path to an inpainted HERA UVH5 data file.

    Returns
    -------
    out_num, out_denom, out_times : tuple of dicts
        Per-source dicts of ``{pol: array}`` for the weighted visibility
        numerator, the weight denominator, and the time axis respectively.
    """
    hd = io.HERAData(filepath)

    out_num   = {sn: {} for sn in best_fit_source_params}
    out_denom = {sn: {} for sn in best_fit_source_params}
    out_times = {sn: {} for sn in best_fit_source_params}

    for source_name, source_info in best_fit_source_params.items():
        lst_center = source_info["right_ascension"] / 15 * np.pi / 12
        lst_range  = (
            lst_center - source_info["lst_half_width"],
            lst_center + source_info["lst_half_width"],
        )

        d, f, n = hd.read(polarizations=["ee", "nn"], lst_range=lst_range)

        l0, m0, n0 = polfilt.radec_to_azalt(
            source_info["right_ascension"],
            source_info["declination"],
            d.times,
            hd.telescope.location,
        )
        lmn0 = np.stack([l0, m0, n0])  # (3, n_times)

        ap     = list(d.antpairs())[0]  # single baseline per file
        bl_vec = hd.antpos[ap[1]] - hd.antpos[ap[0]]
        # Geometric phase toward the source: (n_times, n_freqs)
        phase  = (bl_vec @ lmn0 / constants.c.value * d.freqs[:, None]).T
        phasor = np.exp(-2j * np.pi * phase)

        for pol in ["ee", "nn"]:
            key     = ap + (pol,)
            vis_bl  = d[key]
            flag_bl = f[key]
            nsamp   = n[key]

            w      = np.where(flag_bl, 0.0, nsamp)
            w      = np.where(np.isfinite(vis_bl), w, 0.0)
            vis_bl = np.where(np.isfinite(vis_bl), vis_bl, 0.0)

            out_num[source_name][pol]   = w * vis_bl * phasor
            out_denom[source_name][pol] = w

        out_times[source_name] = d.times

    return out_num, out_denom, out_times


def load_parallel_average_over_baseline(files, max_workers=8):
    """Load files in parallel and form a baseline-averaged phase-rotated dataset.

    Accumulates weighted visibility numerators and denominators across all
    baselines, then divides to obtain the average phase-rotated spectrum for
    each source.

    Parameters
    ----------
    files : list of str
        Inpainted single-baseline UVH5 files to load.
    max_workers : int
        Number of parallel worker processes.

    Returns
    -------
    data_list, flags_list, nsamples_list, times : tuple of dicts
        Per-source averaged visibilities (ee/nn), combined flags, accumulated
        sample weights, and time arrays.
    """
    num   = None
    denom = None
    times = None

    with Pool(processes=max_workers) as pool:
        results = pool.imap_unordered(
            _process_file_hera_with_phasor, files, chunksize=1
        )
        for result in tqdm(results, total=len(files)):
            if result is None:
                continue
            try:
                out_num, out_denom, out_times = result
                if num is None:
                    num, denom, times = out_num, out_denom, out_times
                else:
                    for source_name in best_fit_source_params:
                        for pol in ["ee", "nn"]:
                            num[source_name][pol]   += out_num[source_name][pol]
                            denom[source_name][pol] += out_denom[source_name][pol]
            except Exception as exc:
                print(f"Merge error: {exc}")
                raise

    data_list     = {}
    flags_list    = {}
    nsamples_list = {}

    for source_name in best_fit_source_params:
        d = denom[source_name]
        n = num[source_name]
        data_list[source_name]     = {pol: np.where(d[pol] > 0, n[pol] / d[pol], 0.0) for pol in ["ee", "nn"]}
        flags_list[source_name]    = {pol: d[pol] == 0                                  for pol in ["ee", "nn"]}
        nsamples_list[source_name] = {pol: d[pol]                                        for pol in ["ee", "nn"]}

    return data_list, flags_list, nsamples_list, times

In [ ]:
avg_data_list, avg_flags_list, avg_nsamples_list, avg_times = load_parallel_average_over_baseline(
    load_files, max_workers=MAX_WORKERS
)

In [ ]:
def _dpss_fit(data_2d, weights, time_basis, freq_basis, solver="pinv", return_beta=False):
    """Fit a separable DPSS model to a 2-D (time x frequency) visibility array.

    Solves the weighted least-squares problem

        min_beta  || W^(1/2) (y - (T x F) beta) ||^2

    via a pseudo-inverse, where *T* and *F* are the time and frequency DPSS
    basis matrices respectively.

    Parameters
    ----------
    data_2d : ndarray, shape (n_times, n_freqs)
        Complex visibility array.
    weights : ndarray, shape (n_times, n_freqs)
        Real weight array.
    time_basis : ndarray, shape (n_times, n_time_comps)
        DPSS basis matrix in the time direction.
    freq_basis : ndarray, shape (n_freqs, n_freq_comps)
        DPSS basis matrix in the frequency direction.
    solver : str
        Linear solver passed to ``_linear_fit`` (default: ``"pinv"``)

    Returns
    -------
    ndarray, shape (n_times, n_freqs)
        Model visibilities.
    """
    XTX = np.einsum(
        "ti,fj,tf,tm,fn->ijmn",
        time_basis.conj(), freq_basis.conj(),
        weights,
        time_basis,       freq_basis,
        optimize=True,
    )
    n_comps = time_basis.shape[-1] * freq_basis.shape[-1]
    XTX     = XTX.reshape(n_comps, n_comps)

    # X^T W y via the Kronecker identity: (A x B) vec(Y) = A Y B^T
    XTWy = np.ravel(time_basis.conj().T @ (data_2d * weights) @ freq_basis.conj())

    beta, _ = _linear_fit(XTX, XTWy, solver=solver)
    beta    = beta.reshape(time_basis.shape[-1], freq_basis.shape[-1])

    if return_beta:
        return time_basis @ beta @ freq_basis.T, beta
    else:
        return time_basis @ beta @ freq_basis.T


def fit_pulsar_models(
    hd,
    avg_data_list,
    avg_flags_list,
    avg_nsamples_list,
    avg_times,
    temporal_half_width=0.25,
    spectral_half_width=50e-9,
    scintillation_scale=5.0,
    eigenval_cutoff=1e-12,
    foreground_half_width=500e-9,
):
    """Fit a Faraday-rotating DPSS model to each source in ``best_fit_source_params``.

    For every source the function:

    1. Applies a per-band DPSS foreground filter to the ``ee`` and ``nn``
       visibilities to remove spectrally-smooth foreground contamination.
    2. Constructs separable DPSS bases in time and frequency.
    3. Fits left- and right-circular Faraday phasor components and combines
       them into a full polarised source model.

    Parameters
    ----------
    hd : HERAData
        Open data object used for frequency and LST metadata.
    avg_data_list, avg_flags_list, avg_nsamples_list : dict
        Per-source, per-polarisation averaged visibilities, flags, and weights.
    avg_times : dict
        Per-source arrays of UTC times (JD) for the time axis.
    temporal_half_width : float
        DPSS half-bandwidth in the time direction (mHz; default 0.25).
    spectral_half_width : float
        DPSS half-bandwidth in the frequency direction (seconds; default 50 ns).
    scintillation_scale : float
        DPSS half-bandwidth in the time direction for modeling scintillation (mHz; default 5.0)
    eigenval_cutoff : float
        Eigenvalue threshold for DPSS mode truncation.
    foreground_half_width : float
        Delay half-width (seconds) of the foreground DPSS filter applied
        before modelling.

    Returns
    -------
    dict
        Nested dict ``{source_name: {pol: {"model", "filtered", "wgts"}}}``
        where each value is a full-night array of shape ``(n_times, n_freqs)``.
    """
    results = {}

    for source_name, source_info in best_fit_source_params.items():
        # Build LST mask and per-polarisation weights
        lst_center = source_info["right_ascension"] / 15 * np.pi / 12
        lst_range  = (
            lst_center - source_info["lst_half_width"],
            lst_center + source_info["lst_half_width"],
        )
        mask = (lst_range[0] < hd.lsts) & (hd.lsts < lst_range[1])

        data_ee = avg_data_list[source_name]["ee"]
        data_nn = avg_data_list[source_name]["nn"]

        flags = (
            avg_flags_list[source_name]["ee"] | prior_flags[mask, :] |
            avg_flags_list[source_name]["nn"]
        )

        # Split into low/high sub-bands around the FM gap
        _, FILTER_BANDS = flag_utils.get_minimal_slices(
            flags, freqs=hd.freqs,
            freq_cuts=[(FM_LOW_FREQ + FM_HIGH_FREQ) * 0.5e6],
        )

        n_ee = np.where(
            avg_flags_list[source_name]["ee"] | prior_flags[mask, :],
            0.0, avg_nsamples_list[source_name]["ee"],
        )
        n_nn = np.where(
            avg_flags_list[source_name]["nn"] | prior_flags[mask, :],
            0.0, avg_nsamples_list[source_name]["nn"],
        )

        times_msec = avg_times[source_name] * 24 * 3600 / 1e3

        # Per-band DPSS foreground filter
        data_ee_filt = np.zeros_like(data_ee)
        data_nn_filt = np.zeros_like(data_nn)

        for band in FILTER_BANDS:
            filter_kwargs = dict(
                filter_centers=[0],
                filter_half_widths=[foreground_half_width],
                eigenval_cutoff=[eigenval_cutoff],
                mode="dpss_solve",
                ridge_alpha=eigenval_cutoff,
                max_contiguous_edge_flags=hd.freqs[band].size,
            )
            _, data_ee_filt[:, band], _ = dspec.fourier_filter(
                hd.freqs[band], data_ee[:, band], n_ee[:, band], **filter_kwargs
            )
            _, data_nn_filt[:, band], _ = dspec.fourier_filter(
                hd.freqs[band], data_nn[:, band], n_nn[:, band], **filter_kwargs
            )
        
        # Allocate output arrays (full-night, full-band)
        full_rm_model_ee    = np.zeros((hd.times.shape[0], hd.freqs.shape[0]), dtype=complex)
        full_rm_model_nn    = np.zeros((hd.times.shape[0], hd.freqs.shape[0]), dtype=complex)
        full_scint_model_ee    = np.zeros((hd.times.shape[0], hd.freqs.shape[0]), dtype=complex)
        full_scint_model_nn    = np.zeros((hd.times.shape[0], hd.freqs.shape[0]), dtype=complex)
        full_fg_filtered_ee = np.zeros((hd.times.shape[0], hd.freqs.shape[0]), dtype=complex)
        full_fg_filtered_nn = np.zeros((hd.times.shape[0], hd.freqs.shape[0]), dtype=complex)
        full_wgts_ee     = np.zeros((hd.times.shape[0], hd.freqs.shape[0]), dtype=complex)
        full_wgts_nn     = np.zeros((hd.times.shape[0], hd.freqs.shape[0]), dtype=complex)
        residual_ee     = np.zeros((hd.times.shape[0], hd.freqs.shape[0]), dtype=complex)
        residual_nn     = np.zeros((hd.times.shape[0], hd.freqs.shape[0]), dtype=complex)

        if source_info["fit_scintillation"]:
            scintillation_hw = 1e3 / source_info["scintillation_timescale"]
            time_filter_kwargs = dict(
                filter_centers=[0],
                filter_half_widths=[scintillation_hw],
                eigenval_cutoff=[eigenval_cutoff],
                mode="dpss_solve",
                ridge_alpha=eigenval_cutoff,
                max_contiguous_edge_flags=times_msec.size,
                filter_dims=0
            )
        else:
            time_filter_kwargs = {}

        for band in FILTER_BANDS:
            time_basis = dspec.dpss_operator(
                times_msec, [0], [temporal_half_width], eigenval_cutoff=[eigenval_cutoff]
            )[0]
            freq_basis = dspec.dpss_operator(
                hd.freqs[band], [0], [spectral_half_width], eigenval_cutoff=[eigenval_cutoff]
            )[0]

            # Faraday phasor  exp(2i lambda^2 RM)  for this sub-band
            rm_phasor = np.exp(
                2j * (constants.c.value / hd.freqs[band]) ** 2
                * source_info["rotation_measure"]
            )

            # Extract data
            fg_filt_ee = data_ee_filt[:, band]
            fg_filt_nn = data_nn_filt[:, band]
            w_ee = n_ee[:, band]
            w_nn = n_nn[:, band]
            visible = (w_ee > 0).astype(float), (w_nn > 0).astype(float)

            # Fit left- and right-circular components independently,
            # then reconstruct the full Faraday-modulated model.
            # Each data array is de-rotated by +/-RM before fitting so that
            # the DPSS basis sees a spectrally smooth signal.
            p_right_ee = _dpss_fit(fg_filt_ee * rm_phasor.conj(), w_ee, time_basis, freq_basis)
            p_left_ee  = _dpss_fit(fg_filt_ee * rm_phasor,        w_ee, time_basis, freq_basis)
            p_right_nn = _dpss_fit(fg_filt_nn * rm_phasor.conj(), w_nn, time_basis, freq_basis)
            p_left_nn  = _dpss_fit(fg_filt_nn * rm_phasor,        w_nn, time_basis, freq_basis)
    
            # Sum the individual contributions to form RM model        
            rm_model_ee = p_left_ee * rm_phasor.conj() + p_right_ee * rm_phasor
            rm_model_nn = p_left_nn * rm_phasor.conj() + p_right_nn * rm_phasor

            if source_info["fit_scintillation"]:
                scint_model_ee, _, _ = dspec.fourier_filter(
                    times_msec,
                    (fg_filt_ee - rm_model_ee),
                    w_ee,
                    **time_filter_kwargs
                )
                scint_model_nn, _, _ = dspec.fourier_filter(
                    times_msec,
                    (fg_filt_nn - rm_model_nn),
                    w_nn,
                    **time_filter_kwargs
                )
            else:
                scint_model_ee = np.zeros_like(fg_filt_ee)
                scint_model_nn = np.zeros_like(fg_filt_nn)

            # Store results in arrays the size of the data
            full_rm_model_ee[mask, band] = rm_model_ee
            full_rm_model_nn[mask, band] = rm_model_nn
            full_scint_model_ee[mask, band] = scint_model_ee
            full_scint_model_nn[mask, band] = scint_model_nn
            full_fg_filtered_ee[mask, band] = fg_filt_ee
            full_fg_filtered_nn[mask, band] = fg_filt_nn
            full_wgts_ee[mask, band] = w_ee
            full_wgts_nn[mask, band] = w_nn
            residual_ee[mask, band] = fg_filt_ee - rm_model_ee - scint_model_ee
            residual_nn[mask, band] = fg_filt_nn - rm_model_nn - scint_model_nn

        results[source_name] = {
            "ee": {
                "rm_model": full_rm_model_ee, 
                "fg_filtered": full_fg_filtered_ee, 
                "scint_model": full_scint_model_ee, 
                "residual": residual_ee, 
                "wgts": full_wgts_ee
            },
            "nn": {
                "rm_model": full_rm_model_nn, 
                "fg_filtered": full_fg_filtered_nn,
                "scint_model": full_scint_model_nn,
                "residual": residual_nn, 
                "wgts": full_wgts_nn
            },
        }

    return results

In [ ]:
results = fit_pulsar_models(
    hd_meta,
    avg_data_list,
    avg_flags_list,
    avg_nsamples_list,
    avg_times,
    temporal_half_width=MODEL_FRATE_HALF_WIDTH,
    spectral_half_width=MODEL_DELAY_HALF_WIDTH,
    eigenval_cutoff=EIGENVAL_CUTOFF,
    foreground_half_width=FG_DELAY_HALF_WIDTH,
)

In [ ]:
# Get autos for plotting here
auto_hd = io.HERAData(auto_file)
auto_data, _, _ = auto_hd.read()
auto_bl = auto_hd.antpairs[0]
df, dt = np.median(np.diff(auto_hd.freqs)), np.median(np.diff(auto_hd.times)) * 24 * 3600

# *Figure 3: Polarized Source Model vs. Data (Low Band)*

In [ ]:
for source_name in results:
    # Safe per-pixel normalisation: divide by the product of the ee/nn weight masks
    lst_center = best_fit_source_params[source_name]["right_ascension"] / 15 * np.pi / 12
    lst_range  = (
        lst_center - best_fit_source_params[source_name]["lst_half_width"],
        lst_center + best_fit_source_params[source_name]["lst_half_width"],
    )
    mask = (lst_range[0] < hd_meta.lsts) & (hd_meta.lsts < lst_range[1])
    extent = [hd_meta.freqs[0] / 1e6, hd_meta.freqs[-1] / 1e6, lst_array[mask][-1] * 12 / np.pi, lst_array[mask][0] * 12 / np.pi]
    
    w_ee = results[source_name]["ee"]["wgts"]
    w_nn = results[source_name]["nn"]["wgts"]
    w_eff = (w_ee ** -1 + w_nn ** -1) ** -1
    noise_est = (auto_data[auto_bl + ('ee',)] + auto_data[auto_bl + ('nn',)]) / np.sqrt(4 * df * dt * w_eff.real)
    norm = np.where((w_ee > 0) & (w_nn > 0), np.abs(noise_est), np.nan)

    pQ_data  = results[source_name]["ee"]["fg_filtered"] - results[source_name]["nn"]["fg_filtered"]
    pQ_rm_model = results[source_name]["ee"]["rm_model"] - results[source_name]["nn"]["rm_model"]
    pQ_scint_model = results[source_name]["ee"]["scint_model"] - results[source_name]["nn"]["scint_model"]
    pQ_model = pQ_rm_model + pQ_scint_model
    pQ_resid = pQ_data - pQ_model

    pI_data  = results[source_name]["ee"]["fg_filtered"] + results[source_name]["nn"]["fg_filtered"]
    pI_rm_model = results[source_name]["ee"]["rm_model"] + results[source_name]["nn"]["rm_model"]
    pI_scint_model = results[source_name]["ee"]["scint_model"] + results[source_name]["nn"]["scint_model"]
    pI_model = pI_rm_model + pI_scint_model
    pI_resid = pI_data - pI_model

    vmin, vmax = 0, 3.0

    fig, axs = plt.subplots(1, 4, figsize=(18, 5), sharey=True, sharex=True)
    im_kwargs = dict(cmap='turbo', aspect='auto', interpolation='None', vmin=vmin, vmax=vmax, extent=extent)

    axs[0].imshow(np.abs(pQ_data)[mask]  / norm[mask], **im_kwargs)
    axs[0].set_title(f'{source_name} -- Foreground-Filtered pI Data')

    axs[1].imshow(np.abs(pQ_rm_model)[mask] / norm[mask], **im_kwargs)
    axs[1].set_title(f'{source_name} -- Faraday Model')

    axs[2].imshow(np.abs(pQ_scint_model)[mask] / norm[mask], **im_kwargs)
    axs[2].set_title(f'{source_name} -- Scintillation Model')

    im = axs[3].imshow(np.abs(pQ_resid)[mask] / norm[mask], **im_kwargs)
    axs[3].set_title(f'{source_name} -- Residual (Data - Model)')

    for ax in axs:
        ax.set_xlabel('Frequency (MHz)')
        #ax.set_ylabel('Time index')
    plt.xlim([45, 90])
    plt.tight_layout()
    plt.colorbar(im, ax=axs, label='|pQ| (SNR)', extend='max', pad=0.01)
    display(fig)
    plt.close(fig)

    fig, axs = plt.subplots(1, 4, figsize=(18, 5), sharey=True, sharex=True)
    im_kwargs = dict(cmap='turbo', aspect='auto', interpolation='None', vmin=vmin, vmax=vmax, extent=extent)

    axs[0].imshow(np.abs(pI_data)[mask]  / norm[mask], **im_kwargs)
    axs[0].set_title(f'{source_name} -- Foreground-Filtered pI Data')

    axs[1].imshow(np.abs(pI_rm_model)[mask] / norm[mask], **im_kwargs)
    axs[1].set_title(f'{source_name} -- Faraday Model')

    axs[2].imshow(np.abs(pI_scint_model)[mask] / norm[mask], **im_kwargs)
    axs[2].set_title(f'{source_name} -- Scintillation Model')

    im = axs[3].imshow(np.abs(pI_resid)[mask] / norm[mask], **im_kwargs)
    axs[3].set_title(f'{source_name} -- Residual (Data - Model)')

    for ax in axs:
        ax.set_xlabel('Frequency (MHz)')
        #ax.set_ylabel('Time index')
    plt.xlim([45, 90])
    plt.tight_layout()
    plt.colorbar(im, ax=axs, label='|pI| (SNR)', extend='max', pad=0.01)
    display(fig)
    plt.close(fig)

# *Figure 4: Polarized Source Model vs. Data (High Band)*

In [ ]:
for source_name in results:
    # Safe per-pixel normalisation: divide by the product of the ee/nn weight masks
    lst_center = best_fit_source_params[source_name]["right_ascension"] / 15 * np.pi / 12
    lst_range  = (
        lst_center - best_fit_source_params[source_name]["lst_half_width"],
        lst_center + best_fit_source_params[source_name]["lst_half_width"],
    )
    mask = (lst_range[0] < hd_meta.lsts) & (hd_meta.lsts < lst_range[1])
    extent = [hd_meta.freqs[0] / 1e6, hd_meta.freqs[-1] / 1e6, lst_array[mask][-1] * 12 / np.pi, lst_array[mask][0] * 12 / np.pi]
    
    w_ee = results[source_name]["ee"]["wgts"]
    w_nn = results[source_name]["nn"]["wgts"]
    w_eff = (w_ee ** -1 + w_nn ** -1) ** -1
    noise_est = (auto_data[auto_bl + ('ee',)] + auto_data[auto_bl + ('nn',)]) / np.sqrt(4 * df * dt * w_eff.real)
    norm = np.where((w_ee > 0) & (w_nn > 0), np.abs(noise_est), np.nan)

    pQ_data  = results[source_name]["ee"]["fg_filtered"] - results[source_name]["nn"]["fg_filtered"]
    pQ_rm_model = results[source_name]["ee"]["rm_model"] - results[source_name]["nn"]["rm_model"]
    pQ_scint_model = results[source_name]["ee"]["scint_model"] - results[source_name]["nn"]["scint_model"]
    pQ_model = pQ_rm_model + pQ_scint_model
    pQ_resid = pQ_data - pQ_model

    pI_data  = results[source_name]["ee"]["fg_filtered"] + results[source_name]["nn"]["fg_filtered"]
    pI_rm_model = results[source_name]["ee"]["rm_model"] + results[source_name]["nn"]["rm_model"]
    pI_scint_model = results[source_name]["ee"]["scint_model"] + results[source_name]["nn"]["scint_model"]
    pI_model = pI_rm_model + pI_scint_model
    pI_resid = pI_data - pI_model

    vmin, vmax = 0, 3.0

    fig, axs = plt.subplots(1, 4, figsize=(18, 5), sharey=True, sharex=True)
    im_kwargs = dict(cmap='turbo', aspect='auto', interpolation='None', vmin=vmin, vmax=vmax, extent=extent)

    axs[0].imshow(np.abs(pQ_data)[mask]  / norm[mask], **im_kwargs)
    axs[0].set_title(f'{source_name} -- Foreground-Filtered pI Data')

    axs[1].imshow(np.abs(pQ_rm_model)[mask] / norm[mask], **im_kwargs)
    axs[1].set_title(f'{source_name} -- Faraday Model')

    axs[2].imshow(np.abs(pQ_scint_model)[mask] / norm[mask], **im_kwargs)
    axs[2].set_title(f'{source_name} -- Scintillation Model')

    im = axs[3].imshow(np.abs(pQ_resid)[mask] / norm[mask], **im_kwargs)
    axs[3].set_title(f'{source_name} -- Residual (Data - Model)')

    for ax in axs:
        ax.set_xlabel('Frequency (MHz)')
        #ax.set_ylabel('Time index')
    plt.xlim([108, 235])
    plt.tight_layout()
    plt.colorbar(im, ax=axs, label='|pQ| (SNR)', extend='max', pad=0.01)
    display(fig)
    plt.close(fig)

    fig, axs = plt.subplots(1, 4, figsize=(18, 5), sharey=True, sharex=True)
    im_kwargs = dict(cmap='turbo', aspect='auto', interpolation='None', vmin=vmin, vmax=vmax, extent=extent)

    axs[0].imshow(np.abs(pI_data)[mask]  / norm[mask], **im_kwargs)
    axs[0].set_title(f'{source_name} -- Foreground-Filtered pI Data')

    axs[1].imshow(np.abs(pI_rm_model)[mask] / norm[mask], **im_kwargs)
    axs[1].set_title(f'{source_name} -- Faraday Model')

    axs[2].imshow(np.abs(pI_scint_model)[mask] / norm[mask], **im_kwargs)
    axs[2].set_title(f'{source_name} -- Scintillation Model')

    im = axs[3].imshow(np.abs(pI_resid)[mask] / norm[mask], **im_kwargs)
    axs[3].set_title(f'{source_name} -- Residual (Data - Model)')

    for ax in axs:
        ax.set_xlabel('Frequency (MHz)')
        #ax.set_ylabel('Time index')
    plt.xlim([108, 235])
    plt.tight_layout()
    plt.colorbar(im, ax=axs, label='|pI| (SNR)', extend='max', pad=0.01)
    display(fig)
    plt.close(fig)

## Save Models

In [ ]:
if SAVE_MODELS:
    hd_out = io.HERAData(load_files[0])
    hd_out.read(polarizations=["ee", "nn"])
    
    for source_name in best_fit_source_params:
        for model_name in ["rm_model", "scint_model"]:
            #for source_name, source_info in best_fit_source_params.items():
            # Build a minimal HERAData object from the first load file as a template
            fsname = source_name.replace(" ", "_").replace("-", "_")
            source_info = best_fit_source_params[source_name]
            
            # Use the antenna pair in the first file as a dummy baseline
            fake_bl = list(hd_out.antpairs)[0]
            
            dc_out = datacontainer.DataContainer({
                fake_bl + ("ee",): results[source_name]["ee"][model_name],
                fake_bl + ("nn",): results[source_name]["nn"][model_name],
            })
            flags_out = datacontainer.DataContainer({
                fake_bl + ("ee",): results[source_name]["ee"]["wgts"] < 1,
                fake_bl + ("nn",): results[source_name]["nn"]["wgts"] < 1,
            })
            nsamples_out = datacontainer.DataContainer({
                fake_bl + ("ee",): results[source_name]["ee"]["wgts"].real,
                fake_bl + ("nn",): results[source_name]["nn"]["wgts"].real,
            })
                    
            # Attach source params and processing provenance as extra keywords
            hd_out.extra_keywords.update({
                "SOURCE_RA":               source_info["right_ascension"],
                "SOURCE_DEC":              source_info["declination"],
                "SOURCE_RM":               source_info["rotation_measure"],
                "FIT_SCINTILLATION":       source_info["fit_scintillation"],
                "SCINTILLATION_TIMESCALE": source_info["scintillation_timescale"],
                "SOURCE_NAME":             source_name,
                "FG_DELAY_HALF_WIDTH":     FG_DELAY_HALF_WIDTH,
                "MODEL_DELAY_HALF_WIDTH":  MODEL_DELAY_HALF_WIDTH,
                "MODEL_FRATE_HALF_WIDTH":  MODEL_FRATE_HALF_WIDTH,
                "EIGENVAL_CUTOFF":         EIGENVAL_CUTOFF,
            })

            outfile = os.path.join(
                os.path.dirname(CORNER_TURN_MAP_YAML),
                f"phased.{jdstr}.{fsname}.{model_name}.uvh5"
            )
            hd_out.history += add_to_history

            hd_out.update(data=dc_out, flags=flags_out, nsamples=nsamples_out)
            hd_out.write_uvh5(outfile, clobber=True)
            print(f"Wrote {outfile}")

In [ ]:
os.path.dirname(CORNER_TURN_MAP_YAML)

## Metadata

In [ ]:
for repo in ['hera_cal', 'hera_filters', 'pyuvdata', 'numpy']:
    exec(f'from {repo} import __version__')
    print(f'{repo}: {__version__}')

In [ ]:
print(f'Finished execution in {(time.time() - tstart) / 60:.2f} minutes.')